# Findings Explorer

This notebook loads all analysis outputs and produces interactive charts.

**Prerequisites:** Run scripts 01–06 before opening this notebook.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()

PARSED_DIR = Path('../data/parsed')
ANALYSIS_DIR = Path('../data/analysis')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
print('Ready.')

## Load All Outputs

In [ ]:
# Load data
def load_if_exists(path):
    p = Path(path)
    if p.exists():
        return pd.read_csv(p)
    print(f'Not found: {p}')
    return None

cite_df   = load_if_exists(PARSED_DIR / 'citations.csv')
ans_df    = load_if_exists(PARSED_DIR / 'answers.csv')
pages_df  = load_if_exists(PARSED_DIR / 'source_pages.csv')
dom_df    = load_if_exists(ANALYSIS_DIR / 'domain_frequency.csv')
pos_df    = load_if_exists(ANALYSIS_DIR / 'positional_distribution.csv')
cat_df    = load_if_exists(ANALYSIS_DIR / 'category_breakdown.csv')

# Load summary JSON
summary_path = ANALYSIS_DIR / 'summary_stats.json'
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

print('\nSummary Stats:')
for k, v in summary.get('summary', {}).items():
    print(f'  {k}: {v}')

## Chart 1: Positional Distribution

In [ ]:
if pos_df is not None:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(pos_df['position_decile'], pos_df['pct'], color='#4285F4', edgecolor='white')
    ax.set_xlabel('Position in Document (decile)', fontweight='bold')
    ax.set_ylabel('% of Citations', fontweight='bold')
    ax.set_title('Where in a Page Are Cited Sentences Found?', fontsize=13)
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()
    
    # H1 test result
    h1 = summary.get('h1_positional_bias', {})
    if h1:
        print('\nH1 Test Result:')
        print(f"  {h1.get('interpretation', 'N/A')}")
        print(f"  p-value: {h1.get('p_value', 'N/A')}")

## Chart 2: Cited Sentence Length Distribution

In [ ]:
if cite_df is not None:
    lengths = cite_df[cite_df['cited_sentence_word_count'] > 0]['cited_sentence_word_count']
    
    if len(lengths) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(lengths.clip(upper=50), bins=25, color='#4285F4', edgecolor='white', alpha=0.8)
        ax.axvline(lengths.median(), color='#EA4335', linestyle='--', linewidth=2, label=f'Median: {lengths.median():.0f} words')
        ax.axvline(lengths.mean(), color='#FBBC05', linestyle='--', linewidth=2, label=f'Mean: {lengths.mean():.1f} words')
        ax.set_xlabel('Cited Sentence Length (words)', fontweight='bold')
        ax.set_ylabel('Count', fontweight='bold')
        ax.set_title('Distribution of Cited Sentence Length', fontsize=13)
        ax.legend()
        plt.tight_layout()
        plt.show()
        
        print(f'\nN: {len(lengths)}')
        print(f'Mean: {lengths.mean():.1f} words')
        print(f'Median: {lengths.median():.0f} words')
        print(f'Std: {lengths.std():.1f}')
        print(f'25th-75th pct: {lengths.quantile(0.25):.0f}–{lengths.quantile(0.75):.0f} words')

## Chart 3: Top 20 Most-Cited Domains

In [ ]:
if dom_df is not None:
    top20 = dom_df.head(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top20['domain'], top20['citation_count'], color='#4285F4', edgecolor='white')
    ax.set_xlabel('Citation Count', fontweight='bold')
    ax.set_title('Top 20 Most-Cited Domains', fontsize=13)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print('\nTop 10:')
    print(top20[['domain', 'citation_count', 'query_count']].head(10).to_string(index=False))

## Chart 4: Fragment Coverage by Query Category

In [ ]:
if cat_df is not None and 'fragment_coverage' in cat_df.columns:
    cat_sorted = cat_df.sort_values('fragment_coverage')
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(cat_sorted['category'], cat_sorted['fragment_coverage'], color='#34A853', edgecolor='white')
    ax.set_xlabel('#:~:text= Fragment Coverage (%)', fontweight='bold')
    ax.set_title('#:~:text= Coverage by Query Category', fontsize=13)
    ax.axvline(cat_df['fragment_coverage'].mean(), color='#EA4335', linestyle='--', linewidth=1.5, label='Mean')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Platform Comparison

In [ ]:
if cite_df is not None and 'platform' in cite_df.columns:
    platform_stats = cite_df.groupby('platform').agg(
        total_citations=('query', 'count'),
        unique_queries=('query', 'nunique'),
        unique_domains=('domain', 'nunique'),
        fragment_coverage=('has_text_fragment', 'mean'),
        avg_sentence_words=('cited_sentence_word_count', 'mean'),
    ).reset_index()
    platform_stats['fragment_coverage'] = (platform_stats['fragment_coverage'] * 100).round(1)
    platform_stats['avg_sentence_words'] = platform_stats['avg_sentence_words'].round(1)
    print('Platform Comparison:')
    display(platform_stats)